In [1]:
# Standard project bootstrap — loads all shared config, paths, and logging utilities.
# This cell must be run first in every session before any other cells.
import sys
from pathlib import Path

# Locate project root by walking up from cwd until we find the .env file,
# then insert src/ into the path so config and download_log are importable.
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents]
                             if (p / '.env').exists()) / 'src'))

from config import *           # PROJECT_ROOT, RAW_DIR, PROCESSED_DIR, DOWNLOADS_DIR, etc.
from download_log import load_log, update_entry, print_entry, print_stale_sources

log = load_log()               # Load the shared download log for status tracking

## 34 · OECD Trade Facilitation Indicators (TFI) Pipeline
Source: OECD / Compare Your Country (compareyourcountry.org/trade-facilitation)  
Indicator: Average trade facilitation performance (composite of 11 sub-indicators A–K)  
Coverage: ~163 countries, 2017 / 2019 / 2022 (biennial)  
Scale: 0–2 (higher = better trade facilitation governance)  
Access: Manual download from CYC Overview table view → Downloads folder  
Sub-indicators (A–K): deferred to Category-1 PDF batch (2024 edition data + breakdowns)

In [11]:
# ============================================================
# CELL 2 — File detection
# Locates the most recent CYC TFI export in DOWNLOADS_DIR by
# glob pattern (no hardcoded filename — any future re-download
# is picked up automatically).
# Derives data_as_of_date from the year columns in the file
# itself — no manual date entry required.
# ============================================================
import os, glob, re
import pandas as pd

# --- Locate file ---
# CYC exports are named "exportedData*.xlsx" by the site.
pattern = os.path.join(DOWNLOADS_DIR, "exportedData*.xlsx")
candidates = glob.glob(pattern)
assert candidates, (
    f"No CYC TFI export found in {DOWNLOADS_DIR}.\n"
    "Download from: https://www.compareyourcountry.org/trade-facilitation\n"
    "Steps: Overview tab → Change view (table) → download selection → save to Downloads."
)
SRC_FILE = max(candidates, key=os.path.getmtime)   # most recent if multiple
print("Using:", os.path.basename(SRC_FILE))

# --- Read raw sheet ---
raw = pd.read_excel(SRC_FILE, sheet_name="Table", header=None, dtype=str)

# --- Derive latest data year from column headers (row index 2) ---
# Row 2 contains: Country | 2017 | 2019 | 2022 (or whatever years this edition carries).
# Take the rightmost numeric-year value — no hardcoding needed.
header_row = raw.iloc[2].dropna().tolist()
year_cols = [v for v in header_row if re.fullmatch(r"\d{4}", str(v))]
assert year_cols, f"Could not find year columns in header row: {header_row}"
DATA_AS_OF = max(year_cols)                        # latest year present in this edition
print("Year columns found:", year_cols)
print("Data as-of (latest year, derived from file):", DATA_AS_OF)

# --- Extract source timestamp from embedded stamp row ---
# Last row contains: "Data obtained from new-display.compareyourcountry.org
# in YYYY-MM-DD HH:MM:SS GMT" — parse download date automatically.
stamp_row = raw.iloc[-1, 0]
print("Source stamp:", stamp_row)
date_match = re.search(r"(\d{4}-\d{2}-\d{2})", str(stamp_row))
DOWNLOAD_DATE = date_match.group(1) if date_match else "unknown"
print("Download date (derived):", DOWNLOAD_DATE)

Using: exportedData (1).xlsx
Year columns found: ['2017', '2019', '2022']
Data as-of (latest year, derived from file): 2022
Source stamp: Data obtained from new-display.compareyourcountry.org in 2026-06-24 18:05:00 GMT
Download date (derived): 2026-06-24


In [7]:
# ============================================================
# CELL 3 — Clean, reshape, and ISO3 harmonize
# Drops non-country rows (regional/income aggregates marked
# with "–" values), reshapes wide→long, maps source country
# names to ISO3 via pycountry fuzzy lookup + manual OVERRIDES.
# ============================================================
import pycountry
import numpy as np

# --- Read data rows only ---
# Row 0: title, Row 1: blank, Row 2: header, Rows 3-178: data,
# Row 179: blank, Row 180: source stamp. Use row 2 as header.
df_raw = pd.read_excel(SRC_FILE, sheet_name="Table", header=2, dtype=str)

# Column headers may be int or str depending on Excel cell type — normalise to str.
df_raw.columns = ["country_name_source"] + [str(c).strip() for c in df_raw.columns[1:]]
print(f"Raw shape: {df_raw.shape}")
print("Columns:", df_raw.columns.tolist())

# --- Drop non-data rows ---
# Non-country rows have "–" for all year columns, or are NaN (blank/stamp rows).
year_cols = [c for c in df_raw.columns if re.fullmatch(r"\d{4}", str(c))]
df_raw = df_raw.dropna(subset=["country_name_source"])  # drop blank rows
df_raw = df_raw[~df_raw["country_name_source"].str.strip().str.startswith("Data obtained")]  # drop stamp
df_raw = df_raw[~(df_raw[year_cols] == "–").all(axis=1)]  # drop aggregate rows (all "–")
print(f"After dropping non-country rows: {df_raw.shape}")

# Rows with partial missing years are kept (e.g. Iceland missing 2017);
# NaN obs are dropped later after melt.
print("Rows with any missing year (kept, NaN dropped after melt):",
      df_raw[df_raw[year_cols].isin(["–"]).any(axis=1)]["country_name_source"].tolist())

# --- Replace "–" with NaN and cast to numeric ---
for c in year_cols:
    df_raw[c] = pd.to_numeric(df_raw[c].replace("–", np.nan), errors="coerce")

# --- Reshape wide → long tidy format ---
df_long = df_raw.melt(
    id_vars=["country_name_source"],
    value_vars=year_cols,
    var_name="year",
    value_name="tfi_avg"
)
df_long["year"] = df_long["year"].astype(int)
df_long = df_long.dropna(subset=["tfi_avg"])   # drop missing obs (e.g. Iceland 2017)
print(f"Long format shape: {df_long.shape}")

# --- ISO3 harmonization ---
# Manual overrides for names pycountry cannot fuzzy-match reliably.
# ⚠️ USER INPUT REQUIRED: if new editions introduce new country names
# that fail ISO3 lookup, add them to this dict.
OVERRIDES = {
    "Chinese Taipei":        "TWN",
    "Hong Kong, China":      "HKG",
    "DRC":                   "COD",
    "Lao PDR":               "LAO",
    "Czechia":               "CZE",
    "Türkiye":               "TUR",
    "Côte d'Ivoire":         "CIV",
    "Viet Nam":              "VNM",
    "Russia":                "RUS",
    "Congo":                 "COG",
    "Korea":                 "KOR",
    "Slovak Republic":       "SVK",
}

def to_iso3(name):
    """Resolve a source country name to ISO3 alpha-3 code.
    Uses manual OVERRIDES first, then pycountry fuzzy search."""
    if name in OVERRIDES:
        return OVERRIDES[name]
    try:
        return pycountry.countries.lookup(name).alpha_3
    except LookupError:
        try:
            results = pycountry.countries.search_fuzzy(name)
            return results[0].alpha_3 if results else None
        except Exception:
            return None

df_long["iso3"] = df_long["country_name_source"].map(to_iso3)

# --- Validation: flag unresolved names ---
unresolved = df_long[df_long["iso3"].isna()]["country_name_source"].unique()
if len(unresolved):
    print(f"\n⚠️  UNRESOLVED ISO3 ({len(unresolved)}):")
    for n in sorted(unresolved):
        print(f"  '{n}'")
else:
    print("\n✅ All country names resolved to ISO3")

print(f"\nISO3-resolved rows: {df_long['iso3'].notna().sum()} / {len(df_long)}")

Raw shape: (178, 4)
Columns: ['country_name_source', '2017', '2019', '2022']
After dropping non-country rows: (164, 4)
Dropped as aggregates: ['Iceland']
Long format shape: (491, 3)

✅ All country names resolved to ISO3

ISO3-resolved rows: 491 / 491


In [12]:
# ============================================================
# CELL 4 — Final assembly, validation, and save
# Reorders columns to standard tidy layout, runs validation
# checks, writes tfi_clean.csv to PROCESSED_DIR, and updates
# the download log on confirmed successful save.
# ============================================================

# --- Assemble final dataframe in standard column order ---
tfi_clean = df_long[["iso3", "country_name_source", "year", "tfi_avg"]].copy()
tfi_clean = tfi_clean.sort_values(["iso3", "year"]).reset_index(drop=True)

# --- Validation checks ---
print("=== VALIDATION ===")

# 1. Shape
print(f"Shape: {tfi_clean.shape}")

# 2. No nulls in key columns
nulls = tfi_clean[["iso3", "year", "tfi_avg"]].isnull().sum()
print(f"Nulls:\n{nulls}")

# 3. Score range: all values should be 0–2
out_of_range = tfi_clean[(tfi_clean["tfi_avg"] < 0) | (tfi_clean["tfi_avg"] > 2)]
print(f"Out-of-range scores (expect 0): {len(out_of_range)}")

# 4. Year coverage
print(f"Years present: {sorted(tfi_clean['year'].unique())}")

# 5. Country count per year
print(f"Countries per year:\n{tfi_clean.groupby('year')['iso3'].count()}")

# 6. Spot checks — known anchors (high/mid/low performers)
anchors = {"NLD": "Netherlands", "SGP": "Singapore", "YEM": "Yemen", "TCD": "Chad"}
print("\nSpot checks (2022):")
for iso3, name in anchors.items():
    row = tfi_clean[(tfi_clean["iso3"] == iso3) & (tfi_clean["year"] == 2022)]
    val = row["tfi_avg"].values[0] if len(row) else "NOT FOUND"
    print(f"  {iso3} ({name}): {val}")

# 7. No duplicate iso3+year combinations
dups = tfi_clean[tfi_clean.duplicated(["iso3", "year"])]
print(f"\nDuplicate iso3+year rows (expect 0): {len(dups)}")

# --- Save ---
OUT_FILE = os.path.join(PROCESSED_DIR, "tfi_clean.csv")
tfi_clean.to_csv(OUT_FILE, index=False)
print(f"\n✅ Saved: {OUT_FILE}")
print(f"   Shape: {tfi_clean.shape}")
print(f"   Columns: {tfi_clean.columns.tolist()}")

# --- Update download log (after confirmed save) ---
update_entry(
    "OECD_TFI",
    last_successful_download_date=DOWNLOAD_DATE,
    data_as_of_date=DATA_AS_OF,
    latest_available_version=f"{DATA_AS_OF} edition (CYC); next edition pending PDF batch",
    local_filename=os.path.basename(SRC_FILE),
    notes="Manual download from CYC Overview table view. Sub-indicators A-K deferred to Cat-1 PDF batch."
)
print_entry("OECD_TFI")

=== VALIDATION ===
Shape: (491, 4)
Nulls:
iso3       0
year       0
tfi_avg    0
dtype: int64
Out-of-range scores (expect 0): 0
Years present: [np.int64(2017), np.int64(2019), np.int64(2022)]
Countries per year:
year
2017    163
2019    164
2022    164
Name: iso3, dtype: int64

Spot checks (2022):
  NLD (Netherlands): 1.874
  SGP (Singapore): 1.849
  YEM (Yemen): 0.308
  TCD (Chad): 0.429

Duplicate iso3+year rows (expect 0): 0

✅ Saved: C:\Users\mjbou\governance-framework\data\processed\tfi_clean.csv
   Shape: (491, 4)
   Columns: ['iso3', 'country_name_source', 'year', 'tfi_avg']
[download_log] Updated entry for OECD_TFI
  source_id: OECD_TFI
  last_attempted_date: 2026-06-24
  last_successful_download_date: 2026-06-24
  data_as_of_date: 2022
  local_filename: exportedData (1).xlsx
  latest_available_version: 2022 edition (CYC); next edition pending PDF batch
  no_update_reason: nan
  notes: Manual download from CYC Overview table view. Sub-indicators A-K deferred to Cat-1 PDF batch.

In [13]:
# ============================================================
# CELL 5 — Pipeline summary
# Confirms what was built, coverage, and output location.
# ============================================================
print("=" * 55)
print("TFI PIPELINE COMPLETE")
print("=" * 55)
print(f"Source:       OECD TFI via Compare Your Country (CYC)")
print(f"Edition:      {DATA_AS_OF} (downloaded {DOWNLOAD_DATE})")
print(f"Countries:    {tfi_clean['iso3'].nunique()}")
print(f"Years:        {sorted(tfi_clean['year'].unique())}")
print(f"Observations: {len(tfi_clean)}")
print(f"Output:       {OUT_FILE}")
print(f"Metric:       tfi_avg (composite average, 0–2 scale)")
print(f"Deferred:     A–K sub-indicators → Cat-1 PDF batch")
print(f"              2024 edition data → Cat-1 PDF batch")
print("=" * 55)

TFI PIPELINE COMPLETE
Source:       OECD TFI via Compare Your Country (CYC)
Edition:      2022 (downloaded 2026-06-24)
Countries:    164
Years:        [np.int64(2017), np.int64(2019), np.int64(2022)]
Observations: 491
Output:       C:\Users\mjbou\governance-framework\data\processed\tfi_clean.csv
Metric:       tfi_avg (composite average, 0–2 scale)
Deferred:     A–K sub-indicators → Cat-1 PDF batch
              2024 edition data → Cat-1 PDF batch
